# Small-architecture five-seed run
Uses the transferred five-fold data, writes outputs to `/home/jovyan/thesis_runs`, and includes the original binary metric helpers required by the training loop.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")
CODE_ROOT = Path("/home/jovyan/thesis_code")
OUTPUT_ROOT = Path("/home/jovyan/thesis_runs")
WORKER = CODE_ROOT / "ADNI_JHub_SmallArch_Worker.py"

SEEDS = [17, 42, 73, 101, 137]
FOLDS = [0, 1, 2, 3, 4]

MAX_EPOCHS = 50
BATCH_SIZE = 2

EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_small_v1_md050"
# Do not write into the transferred dataset tree: it may be read-only.
EXPERIMENT_ROOT = OUTPUT_ROOT / EXPERIMENT_NAME

print("Project root:", PROJECT_ROOT)
print("Worker:", WORKER)
print("Writable output root:", OUTPUT_ROOT)
print("Seeds:", SEEDS)
print("Folds:", FOLDS)
print("Total runs:", len(SEEDS) * len(FOLDS))


In [ ]:
# Input check: fail before training if the transferred inputs or worker are missing.

assert PROJECT_ROOT.exists(), f"Missing project root: {PROJECT_ROOT}"
assert WORKER.exists(), (
    f"Missing worker script: {WORKER}\n"
    "Upload ADNI_JHub_SmallArch_Worker.py into /home/jovyan/thesis_code."
)

input_root = PROJECT_ROOT / "models" / "final_task_ready_inputs" / "mci_prognosis"
missing_folds = []

for fold in FOLDS:
    path = input_root / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"
    if not path.exists():
        missing_folds.append(str(path))

if missing_folds:
    raise FileNotFoundError(
        "Missing prepared fold CSVs:\n" + "\n".join(missing_folds)
    )

print("All five prepared fold CSVs are present.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
write_test = OUTPUT_ROOT / ".write_test"
write_test.write_text("ok", encoding="utf-8")
write_test.unlink()
print("Writable output directory is ready:", OUTPUT_ROOT)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError(
        "CUDA is not available in this JHub kernel/session. "
        "Do not start the 25-run experiment on CPU."
    )

usage = os.statvfs(str(PROJECT_ROOT))
free_gb = usage.f_bavail * usage.f_frsize / (1024**3)
print(f"Free filesystem space: {free_gb:.1f} GB")

print("\nInput check passed.")


## Run all five seeds × five folds

This cell is restart-safe:

- completed seed/fold runs are skipped;
- an interrupted run with a saved last-epoch checkpoint is resumed;
- a run with no prior checkpoint starts fresh.

Outputs are stored separately under:

`/home/jovyan/thesis_runs/gated_cmt_fixed_equal_fusion_small_v1_md050/seed_<seed>/mci_prognosis/fold_<fold>/`


In [ ]:
for seed in SEEDS:
    for fold in FOLDS:
        fold_root = (
            EXPERIMENT_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
        )

        test_metrics_path = (
            fold_root
            / "predictions"
            / "test_metrics.json"
        )

        last_checkpoint_path = (
            fold_root
            / "checkpoints"
            / "last_epoch_checkpoint.pt"
        )

        if test_metrics_path.exists():
            print("=" * 88)
            print(f"SKIP completed run | seed={seed} fold={fold}")
            print(test_metrics_path)
            continue

        mode = "resume" if last_checkpoint_path.exists() else "fresh"

        print("\n" + "=" * 88)
        print(f"START | seed={seed} fold={fold} mode={mode}")
        print("=" * 88)

        command = [
            sys.executable,
            "-u",
            str(WORKER),
            "--seed", str(seed),
            "--fold", str(fold),
            "--mode", mode,
            "--project-root", str(PROJECT_ROOT),
            "--output-root", str(OUTPUT_ROOT),
            "--max-epochs", str(MAX_EPOCHS),
            "--batch-size", str(BATCH_SIZE),
        ]

        result = subprocess.run(
            command,
            cwd=str(CODE_ROOT),
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"Run failed for seed={seed}, fold={fold}. "
                "The launcher stopped so the problem can be fixed "
                "without starting later runs."
            )

print("\nAll requested seed/fold runs are complete.")


## Aggregate the 25 runs


In [ ]:
# Collect fold metrics and pooled out-of-fold predictions.

fold_rows = []
seed_rows = []

for seed in SEEDS:
    seed_prediction_tables = []

    for fold in FOLDS:
        fold_root = (
            EXPERIMENT_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
        )

        metrics_path = fold_root / "predictions" / "test_metrics.json"
        predictions_path = fold_root / "predictions" / "test_predictions.csv"

        if not metrics_path.exists():
            fold_rows.append({
                "seed": seed,
                "fold": fold,
                "status": "missing",
            })
            continue

        with metrics_path.open("r", encoding="utf-8") as f:
            metrics = json.load(f)

        final_metrics = metrics["test_metrics"]["final"]

        fold_rows.append({
            "seed": seed,
            "fold": fold,
            "status": "complete",
            "best_epoch": metrics["checkpoint_epoch"],
            "validation_roc_auc": metrics["checkpoint_validation_auc"],
            "test_roc_auc": final_metrics["roc_auc"],
            "test_average_precision": final_metrics["average_precision"],
            "test_balanced_accuracy": final_metrics["balanced_accuracy"],
            "test_brier_score": final_metrics["brier_score"],
            "test_ece": final_metrics["expected_calibration_error"],
            "test_mean_uncertainty": final_metrics["mean_uncertainty"],
            "test_participants": metrics["test_participants"],
        })

        if predictions_path.exists():
            pred = pd.read_csv(predictions_path)
            pred["seed"] = seed
            pred["fold"] = fold
            seed_prediction_tables.append(pred)

    # Pool all five out-of-fold test prediction files for this seed.
    if len(seed_prediction_tables) == len(FOLDS):
        pooled = pd.concat(seed_prediction_tables, ignore_index=True)

        seed_rows.append({
            "seed": seed,
            "pooled_test_participants": len(pooled),

            # Correct column names from test_predictions.csv
            "pooled_hybrid_roc_auc": roc_auc_score(
                pooled["TARGET"],
                pooled["FINAL_P_pMCI"],
            ),

            "pooled_hybrid_average_precision": average_precision_score(
                pooled["TARGET"],
                pooled["FINAL_P_pMCI"],
            ),

            "mean_fold_roc_auc": np.mean([
                row["test_roc_auc"]
                for row in fold_rows
                if row.get("seed") == seed
                and row.get("status") == "complete"
            ]),
        })

fold_summary = pd.DataFrame(fold_rows)
seed_summary = pd.DataFrame(seed_rows)

print("Fold-level results:")
display(fold_summary)

print("\nSeed-level pooled out-of-fold results:")
display(seed_summary)

if len(seed_summary) == len(SEEDS):

    mean_auc = seed_summary["pooled_hybrid_roc_auc"].mean()
    sd_auc = seed_summary["pooled_hybrid_roc_auc"].std(ddof=1)

    mean_ap = seed_summary["pooled_hybrid_average_precision"].mean()
    sd_ap = seed_summary["pooled_hybrid_average_precision"].std(ddof=1)

    print("\nFive-seed pooled hybrid ROC AUC:")
    print(f"{mean_auc:.6f} ± {sd_auc:.6f}")

    print("\nFive-seed pooled hybrid average precision:")
    print(f"{mean_ap:.6f} ± {sd_ap:.6f}")

    summary_dir = EXPERIMENT_ROOT / "five_seed_summary"
    summary_dir.mkdir(parents=True, exist_ok=True)

    fold_summary.to_csv(
        summary_dir / "fold_metrics_all_seeds.csv",
        index=False,
    )

    seed_summary.to_csv(
        summary_dir / "pooled_metrics_by_seed.csv",
        index=False,
    )

    print("\nSaved summary files to:")
    print(summary_dir)

else:
    print(
        f"\nWARNING: pooled results were available for "
        f"{len(seed_summary)}/{len(SEEDS)} seeds."
    )
